# 01 — Data Collection

Pulling live ADS-B state vectors from the **OpenSky Network** REST API for a defined airspace bounding box, clean the data, and persist it to CSV for downstream notebooks.


## 1. Configuration

Dallas/Fort Worth TRACON (D10) is the default sector — one of the busiest in the US and well-documented for comparison. Swap the bounding box to target any airspace.

> #### Tracon, or **Terminal Radar Approach Control**, is an airport's FAA facility that manages aircraft within a 30-50 mile radius of the airport typically within 10-15 thousand feet.

In [2]:
import sys, os, time, json, requests
import pandas as pd
from datetime import datetime, timezone

sys.path.insert(0, os.path.abspath('../src'))
from airspace import parse_opensky_response, states_to_df

# ── Bounding box: Dallas/Fort Worth TRACON (D10) ──────────────────────────
BBOX = {
    "lamin": 32.0,   # min latitude
    "lamax": 33.5,   # max latitude
    "lomin": -98.0,  # min longitude
    "lomax": -96.0,  # max longitude
}

'''
Example Bounding box for Chicago O'Hare TRACON (C90):
BBOX = {
    "lamin": 39.5,   # min latitude
    "lamax": 42.5,   # max latitude
    "lomin": -88.5,  # min longitude
    "lomax": -87.0,  # max longitude
}
'''

NUM_SNAPSHOTS = 6      # snapshots to collect
SLEEP_SEC   = 10     # OpenSky free tier: 1 req/10s, need to wait between snapshots
OUTPUT_PATH = "../data/raw_states.csv"

print(f"Bounding box: {BBOX}")
print(f"Collecting {NUM_SNAPSHOTS} snapshots (~{NUM_SNAPSHOTS * SLEEP_SEC}s total)...")

Bounding box: {'lamin': 32.0, 'lamax': 33.5, 'lomin': -98.0, 'lomax': -96.0}


#### 1.1 BBOX locator

The cell above uses bbox for defining TRACON locations. You can use the function below to find a bbox for any airport if you know it's icao code.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from get_bbox import main as get_bbox

get_bbox()


Bounding box for KSTL:
lamin: 38.7333378,
lamax: 38.765807,
lomin: -90.4177286,
lomax: -90.335863


## 2. Fetch from OpenSky

The `/states/all` endpoint returns every aircraft currently visible within the bounding box as a list of state vectors.

In [4]:
BASE_URL  = "https://opensky-network.org/api/states/all"
all_states, failed = [], 0

for i in range(NUM_SNAPSHOTS):
    try:
        resp = requests.get(BASE_URL, params=BBOX, timeout=15)
        resp.raise_for_status()
        data     = resp.json()
        snapshot = parse_opensky_response(data)
        all_states.extend(snapshot)
        ts = datetime.fromtimestamp(data.get("time", 0), tz=timezone.utc)
        print(f"  [{i+1}/{NUM_SNAPSHOTS}] {ts.strftime('%H:%M:%S')} UTC — {len(snapshot)} aircraft")
    except Exception as e:
        print(f"  [{i+1}/{NUM_SNAPSHOTS}] FAILED: {e}")
        failed += 1
    if i < NUM_SNAPSHOTS - 1:
        time.sleep(SLEEP_SEC)

print(f"\nTotal state vectors collected: {len(all_states)}  ({failed} snapshots failed)")


  [1/6] 17:16:50 UTC — 146 aircraft
  [2/6] 17:17:00 UTC — 147 aircraft
  [3/6] 17:17:36 UTC — 147 aircraft
  [4/6] 17:17:46 UTC — 148 aircraft
  [5/6] 17:17:54 UTC — 147 aircraft
  [6/6] 17:17:54 UTC — 147 aircraft

Total state vectors collected: 882  (0 snapshots failed)


## 3. Clean & Validate

Drop records with missing critical fields, filter out ground traffic, and apply sanity bounds.

In [5]:
df = states_to_df(all_states)
print(f"Raw shape: {df.shape}")
print("\nNull counts per column:")
print(df.isnull().sum())

Raw shape: (882, 9)

Null counts per column:
icao24          0
callsign        0
time            0
lat             0
lon             0
altitude_ft     0
velocity_kts    0
heading_deg     0
on_ground       0
dtype: int64


In [6]:
df_clean = (df
    .dropna(subset=["lat", "lon", "altitude_ft", "velocity_kts"])
    .query("not on_ground")
    .query("0 < altitude_ft < 60000")
    .query("0 < velocity_kts < 700")
    .query("@BBOX['lamin'] <= lat <= @BBOX['lamax']")
    .query("@BBOX['lomin'] <= lon <= @BBOX['lomax']")
    .copy()
)

df_clean["callsign"]  = df_clean["callsign"].str.strip().replace("", "UNKNOWN")
df_clean["time_utc"]  = pd.to_datetime(df_clean["time"], unit="s", utc=True)
df_clean["alt_band"]  = pd.cut(df_clean["altitude_ft"],
                                bins=[0, 10000, 18000, 28000, 60000],
                                labels=["Low", "Terminal", "Transition", "High"])

print(f"Clean shape: {df_clean.shape}  ({len(df) - len(df_clean)} rows dropped)")
df_clean.head(10)

Clean shape: (872, 11)  (10 rows dropped)


,icao24,callsign,time,lat,lon,altitude_ft,velocity_kts,heading_deg,on_ground,time_utc,alt_band
0,a798ad,WMN589,1.773422e+09,33.2012,-97.9943,27675.000886,357.09336,289.30,False,2026-03-13 17:16:50+00:00,Transition
1,a234ef,ENY4095,1.773422e+09,32.8230,-97.2648,10750.000344,301.88376,339.64,False,2026-03-13 17:16:50+00:00,Terminal
2,a53473,N43408,1.773422e+09,32.3976,-97.8602,4400.000141,96.26688,235.15,False,2026-03-13 17:16:50+00:00,Low
3,abfac6,SWA2531,1.773422e+09,33.4006,-97.0221,7275.000233,300.11472,100.56,False,2026-03-13 17:16:50+00:00,Low
4,a3917e,N329TX,1.773422e+09,32.8596,-97.3200,1450.000046,110.94408,207.95,False,2026-03-13 17:16:50+00:00,Low
5,a6a441,N527PN,1.773422e+09,32.2313,-96.5301,11275.000361,356.76288,319.43,False,2026-03-13 17:16:50+00:00,Terminal
6,a0802e,N131PA,1.773422e+09,32.8428,-96.2271,1525.000049,108.18360,340.56,False,2026-03-13 17:16:50+00:00,Low
7,a48973,N3914Q,1.773422e+09,32.0426,-96.6644,1150.000037,114.85152,343.30,False,2026-03-13 17:16:50+00:00,Low
8,a45052,N3771L,1.773422e+09,32.4769,-96.9001,1300.000042,91.05696,321.24,False,2026-03-13 17:16:50+00:00,Low
9,ad311d,N9491L,1.773422e+09,32.3499,-97.4360,2000.000064,74.70792,242.06,False,2026-03-13 17:16:50+00:00,Low


## 4. Save

In [7]:
os.makedirs("../data", exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_clean)} records -> {OUTPUT_PATH}")
print(f"\nAltitude band distribution:")
print(df_clean["alt_band"].value_counts())

Saved 872 records -> ../data/raw_states.csv

Altitude band distribution:
alt_band
Low           743
Terminal       49
High           44
Transition     36
Name: count, dtype: int64
